<div style="background:linear-gradient(135deg,#143840 0%,#2B6264 100%);border-radius:14px;padding:32px 36px;color:#fff;font-family:'DM Sans',Arial,sans-serif;">
  <div style="font-size:11px;letter-spacing:2px;text-transform:uppercase;color:#FF4B31;font-weight:700;margin-bottom:10px;">Solutions Onboarding &middot; IBOR Training</div>
  <div style="font-size:30px;font-weight:700;line-height:1.15;margin-bottom:10px;">IBOR NB01 &mdash; Portfolio Structure</div>
  <div style="font-size:15px;color:rgba(255,255,255,.82);max-width:640px;line-height:1.55;">Build the eight portfolios (four core plus four from iShares fund data), portfolio groups and derived portfolios, sub-holding keys, and the corporate action source.</div>
</div>

<sub>IBOR pack sequence: NB00 &nbsp;&rarr;&nbsp; **NB01** &nbsp;&rarr;&nbsp; NB02 &nbsp;&rarr;&nbsp; NB03 &nbsp;&rarr;&nbsp; NB04 &nbsp;&rarr;&nbsp; NB05 &nbsp;&rarr;&nbsp; NB06 &nbsp;&rarr;&nbsp; NB07 &nbsp;&rarr;&nbsp; NB08</sub>

# NB01: Portfolio Structure

Creates 8 portfolios (4 original + 4 from iShares data), portfolio groups, derived portfolios, and links corporate action sources.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'lusid-sdk', 'lumipy', '-q'],
                      stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

import os, json
from datetime import datetime, timedelta, timezone
import pandas as pd
import lusid as lu
import lusid.models as lm
import lumipy as lp

pd.set_option("display.max_columns", None)

SCOPE = 'ibor-training-v9'
DATA_DIR = 'data'

def get_factory(app='lusid'):
    secrets_path = "secrets.json"
    with open(secrets_path) as f:
        secrets = json.load(f)
    api_section = secrets.get("api", {})
    pat = api_section.get("accessToken")
    module = __import__(app)
    if pat:
        config_loaders = [module.extensions.ArgsConfigurationLoader(
            api_url=api_section.get("lusidUrl", ""), access_token=pat)]
    else:
        config_loaders = [module.extensions.SecretsFileConfigurationLoader(secrets_path)]
    return module.extensions.SyncApiClientFactory(config_loaders=config_loaders)

def get_lumi():
    secrets_path = "secrets.json"
    with open(secrets_path) as f:
        secrets = json.load(f)
    api_section = secrets.get("api", {})
    pat = api_section.get("accessToken")
    if pat:
        lumi_url = api_section.get("lusidUrl", "").replace("/api", "/honeycomb")
        return lp.get_client(access_token=pat, api_url=lumi_url)
    return lp.get_client(api_secrets_file=secrets_path)

lusid_factory = get_factory('lusid')
lumi = get_lumi()
print('LUSID + Luminesce initialised')

transaction_portfolios_api = lusid_factory.build(lu.TransactionPortfoliosApi)
portfolios_api = lusid_factory.build(lu.PortfoliosApi)
portfolio_groups_api = lusid_factory.build(lu.PortfolioGroupsApi)
cas_api = lusid_factory.build(lu.CorporateActionSourcesApi)

---
## Create Corporate Action Source

In [ ]:
# CA source for stock splits and other corporate actions
try:
    cas_api.create_corporate_action_source(
        create_corporate_action_source_request=lm.CreateCorporateActionSourceRequest(
            scope=SCOPE, code="IBOR-CA-SOURCE-V3",
            display_name="IBOR Training CA Source",
            instrument_scopes=[SCOPE]))
    print("Created CA source: IBOR-CA-SOURCE-V3")
except lu.ApiException as e:
    if 'AlreadyExists' in str(e.body):
        print("CA source already exists")
    else:
        print(f"Error: {str(e.body)[:300]}")

---
## Create Transaction Portfolios

In [ ]:
df_ports = pd.read_csv(f"{DATA_DIR}/ibor_portfolios.csv")
print(f"Creating {len(df_ports)} portfolios...")

for _, row in df_ports.iterrows():
    shks = []
    for col in ['SubHoldingKey1', 'SubHoldingKey2']:
        val = str(row.get(col, '')).strip()
        if val and val != 'nan':
            shks.append(val)
    
    # Portfolio properties
    props = {}
    for prop_code, csv_col in [("Region", "Region"), ("FundType", "FundType"),
                                ("PortfolioManager", "PortfolioManager"),
                                ("Benchmark", "Benchmark"), ("InvestmentStrategy", "InvestmentStrategy")]:
        if pd.notna(row.get(csv_col)) and str(row[csv_col]).strip():
            key = f"Portfolio/{SCOPE}/{prop_code}"
            props[key] = lm.ModelProperty(key=key, value=lm.PropertyValue(label_value=str(row[csv_col]).strip()))
    
    try:
        transaction_portfolios_api.create_portfolio(
            scope=SCOPE,
            create_transaction_portfolio_request=lm.CreateTransactionPortfolioRequest(
                display_name=row['DisplayName'],
                code=row['PortfolioCode'],
                base_currency=row['BaseCurrency'],
                created=datetime(2024, 1, 1, tzinfo=timezone.utc),
                sub_holding_keys=shks,
                corporate_action_source_id=lm.ResourceId(scope=SCOPE, code="IBOR-CA-SOURCE-V3"),
                instrument_scopes=[SCOPE],
                properties=props if props else None
            )
        )
        shk_info = f" SHKs: {len(shks)}" if shks else ""
        print(f"  Created: {row['PortfolioCode']} ({row['DisplayName']}){shk_info}")
    except lu.ApiException as e:
        if 'AlreadyExists' in str(e.body):
            print(f"  Exists: {row['PortfolioCode']}")
        else:
            print(f"  Error: {row['PortfolioCode']}: {str(e.body)[:200]}")

---
## Enable Instrument Event Processing
Set InstrumentEventConfiguration so LUSID generates automatic lifecycle events (bond coupons, IRS cashflows, term deposit maturity).

In [ ]:
# Enable InstrumentEventConfiguration on the derivative/cashflow portfolios.
# The recipe (IBOR-VALUATION-RECIPE) is created later in NB06, so on a fresh scope
# it will not exist yet. That is fine: the IEC recipe link is only needed for
# instrument-event cashflow projection, and NB06 re-establishes this link after it
# creates the recipe. We attempt the link here and skip cleanly if the recipe is absent.
for port_code in ["IBOR-FI", "IBOR-EQ", "IBOR-MA", "IBOR-GAGG"]:
    try:
        transaction_portfolios_api.patch_portfolio_details(
            scope=SCOPE, code=port_code,
            operation=[lm.Operation(
                value={
                    "transactionTemplateScopes": [SCOPE],
                    "recipeId": {"scope": SCOPE, "code": "IBOR-VALUATION-RECIPE"}
                },
                path="/instrumentEventConfiguration", op="add"
            )]
        )
        print(f"  Enabled InstrumentEventConfiguration on {port_code}")
    except lu.ApiException as e:
        body = str(e.body)
        if "IBOR-VALUATION-RECIPE" in body and ("460" in body or "does not exist" in body):
            print(f"  Skipped {port_code}: recipe not created yet (will be set in NB06) - OK")
        else:
            print(f"  Error on {port_code}: {body[:200]}")


---
## Create Portfolio Groups

In [ ]:
groups = [
    {"code": "IBOR-ALL", "name": "All IBOR Portfolios",
     "members": ["IBOR-EQ", "IBOR-FI", "IBOR-MA", "IBOR-CASH", "IBOR-SP500", "IBOR-AITECH", "IBOR-BLKC", "IBOR-GAGG"]},
    {"code": "IBOR-EQUITY-GROUP", "name": "All Equity Portfolios",
     "members": ["IBOR-EQ", "IBOR-SP500", "IBOR-AITECH", "IBOR-BLKC"]},
    {"code": "IBOR-FI-GROUP", "name": "All Fixed Income Portfolios",
     "members": ["IBOR-FI", "IBOR-GAGG"]},
]

for g in groups:
    try:
        portfolio_groups_api.create_portfolio_group(
            scope=SCOPE,
            create_portfolio_group_request=lm.CreatePortfolioGroupRequest(
                code=g['code'], display_name=g['name'],
                values=[lm.ResourceId(scope=SCOPE, code=m) for m in g['members']]
            )
        )
        print(f"  Created group: {g['name']} ({len(g['members'])} portfolios)")
    except lu.ApiException as e:
        if 'AlreadyExists' in str(e.body):
            print(f"  Exists: {g['code']}")
        else:
            print(f"  Error: {g['code']}: {str(e.body)[:200]}")

---
## Create Derived Portfolios (Currency Share Classes)

In [ ]:
derived = [
    {"code": "IBOR-MA-USD", "name": "Multi Asset (USD Share Class)", "parent": "IBOR-MA"},
    {"code": "IBOR-MA-GBP", "name": "Multi Asset (GBP Share Class)", "parent": "IBOR-MA"}
]

api_client = transaction_portfolios_api.api_client
for d in derived:
    try:
        body = {
            "displayName": d["name"],
            "code": d["code"],
            "parentPortfolioId": {"scope": SCOPE, "code": d["parent"]}
        }
        response_data = api_client.call_api(
            f'/api/derivedtransactionportfolios/{SCOPE}', 'POST',
            path_params={}, query_params=[],
            header_params={'Content-Type': 'application/json', 'Accept': 'application/json'},
            body=body, post_params=[], files={},
            response_types_map={},
            auth_settings=['oauth2'], _return_http_data_only=False,
            collection_formats={}, _preload_content=True, _request_timeout=None
        )
        resp_body = json.loads(response_data.raw_data)
        print(f"  Created derived: {d['name']} (type={resp_body.get('type')}, isDerived={resp_body.get('isDerived')})")
    except lu.ApiException as e:
        if 'AlreadyExists' in str(e.body):
            print(f"  Exists: {d['code']}")
        else:
            print(f"  Error: {d['code']}: {str(e.body)[:200]}")

---
## Verification

In [ ]:
query = f"SELECT PortfolioCode, PortfolioType, BaseCurrency FROM Lusid.Portfolio WHERE PortfolioScope = '{SCOPE}' LIMIT 20"
try:
    df = lumi.run(query, quiet=True)
    print(f"Portfolios in scope {SCOPE}:")
    display(df)
except Exception as e:
    print(f"Query error: {e}")

print("\nNB01 Complete. Proceed to NB02: Transaction Type Configuration.")